# build_review_pages_assets.py
 
 Combined from `Archived/python/build_review_pages_assets.py`.

In [1]:
from pathlib import Path
import json
import os
import re
import warnings

import geopandas as gpd
import pandas as pd
import yaml

warnings.filterwarnings("ignore", category=FutureWarning)

try:
    SITE_DIR = Path(__file__).resolve().parents[1]
except NameError:
    SITE_DIR = Path.cwd().resolve()
    if SITE_DIR.name.lower() == "python":
        SITE_DIR = SITE_DIR.parent.parent
CONFIG_PATH = SITE_DIR / "_config.yml"


def load_review_config():
    if not CONFIG_PATH.exists():
        raise FileNotFoundError(f"Config file not found: {CONFIG_PATH}")
    config = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8")) or {}
    review_pages = config.get("review_pages")
    if not isinstance(review_pages, dict):
        raise KeyError(f"{CONFIG_PATH} must contain a review_pages section")
    return review_pages


def resolve_config_path(value, base=SITE_DIR):
    path = Path(value)
    if not path.is_absolute():
        path = base / path
    return path.resolve()


CONFIG = load_review_config()
INPUT_FILES_DIR = resolve_config_path(CONFIG["input_files_dir"])
TAZ_DIR = resolve_config_path(CONFIG["taz_dir"], INPUT_FILES_DIR)
TAZ_SHP = resolve_config_path(CONFIG["taz_shapefile"], INPUT_FILES_DIR)
DISTRICTS_DIR = resolve_config_path(CONFIG["districts_dir"], INPUT_FILES_DIR)
COUNTY_SHP = resolve_config_path(CONFIG["county_shapefile"], INPUT_FILES_DIR)
DOCS_DATA_DIR = Path(os.environ.get("REMM_DOCS_DATA_DIR", resolve_config_path(CONFIG["docs_data_dir"]))).resolve()
SE_DATA_DIR = Path(os.environ.get("REMM_SE_DIR", resolve_config_path(CONFIG["se_data_dir"], INPUT_FILES_DIR))).resolve()
TAZ_GEOJSON_FILE = CONFIG["taz_geojson_file"]
AREA_SQMI_DENOMINATOR = CONFIG["area_square_meters_per_square_mile"]
WEB_MAP_EPSG = CONFIG["web_map_epsg"]
GEO_FIELDS = CONFIG["geography_fields"]
METRICS = CONFIG["metrics"]
PREVIEW_METRIC_ALIASES = CONFIG["preview_metric_aliases"]
PREVIEW_METRICS = list(PREVIEW_METRIC_ALIASES) + ["hh_job_total", "hh_size"]
GEOGRAPHIES = CONFIG["geographies"]
PREVIEW_GEOGRAPHIES = CONFIG["preview_geographies"]
PREVIEW_FILES = CONFIG["preview_files"]
INDEX_HTML = SITE_DIR / "docs" / "index.html"

for geography in GEOGRAPHIES.values():
    geography["source"] = resolve_config_path(geography["source"], INPUT_FILES_DIR)


def clean_columns(df):
    df = df.copy()
    df.columns = [str(col).lstrip(";").strip() for col in df.columns]
    return df


def add_area_sqmi(gdf):
    gdf = gdf.copy()
    area_crs = gdf.estimate_utm_crs() or gdf.crs
    gdf["AREA_SQMI"] = gdf.to_crs(area_crs).geometry.area / AREA_SQMI_DENOMINATOR
    return gdf


def review_config():
    years = []
    for path in sorted(SE_DATA_DIR.glob("SE_*.csv")):
        match = re.search(r"SE_(\d{4})\.csv$", path.name)
        if match:
            years.append(int(match.group(1)))
    if not years:
        raise FileNotFoundError(f"No SE_YYYY.csv files found in {SE_DATA_DIR}")

    return SE_DATA_DIR, min(years), max(years)


def available_years(se_dir, start_year, end_year):
    years = []
    for path in sorted(se_dir.glob("SE_*.csv")):
        match = re.search(r"SE_(\d{4})\.csv$", path.name)
        if match and start_year <= int(match.group(1)) <= end_year:
            years.append(int(match.group(1)))
    if not years:
        raise FileNotFoundError(f"No SE_YYYY.csv files found in {se_dir}")
    return years


def normalize_int_columns(df, columns):
    for col in columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")
    return df


def add_feature_ids(gdf, geography_type, keys, name_col):
    gdf = gdf.copy()
    for key in keys:
        if key not in gdf.columns:
            gdf[key] = pd.NA
    if name_col not in gdf.columns:
        gdf[name_col] = pd.NA
    gdf["geography_type"] = geography_type
    gdf["geography_id"] = gdf[keys].astype(str).agg("_".join, axis=1)
    gdf["geography_name"] = gdf[name_col].fillna(gdf["geography_id"])
    return gdf


def build_taz_geography():
    if not TAZ_SHP.exists():
        raise FileNotFoundError(f"TAZ shapefile not found: {TAZ_SHP}")

    gdf = clean_columns(gpd.read_file(TAZ_SHP))
    for col in GEO_FIELDS:
        if col not in gdf.columns:
            gdf[col] = pd.NA

    int_cols = ["TAZID", "CO_TAZID", "CO_FIPS", "CITY_FIPS", "DISTLRG", "DISTMED", "DISTSML"]
    gdf = normalize_int_columns(gdf, int_cols)
    if gdf.crs is None:
        raise ValueError(f"{TAZ_SHP} does not have a coordinate reference system")

    map_gdf = gdf[GEO_FIELDS + ["geometry"]].copy()
    web_gdf = add_area_sqmi(map_gdf).to_crs(epsg=WEB_MAP_EPSG)
    web_gdf = web_gdf.rename(columns={"TAZID": "taz_id"})
    web_gdf["geography_type"] = "taz"
    web_gdf["geography_id"] = web_gdf["taz_id"].astype(str)
    web_gdf["geography_name"] = "TAZ " + web_gdf["geography_id"]

    xwalk = gdf[GEO_FIELDS].rename(columns={"TAZID": "taz_id"}).drop_duplicates("taz_id")
    return web_gdf, map_gdf, xwalk


def read_boundary_layer(geography_type):
    config = GEOGRAPHIES[geography_type]
    path = config["source"]
    if not path.exists():
        raise FileNotFoundError(f"{geography_type} shapefile not found: {path}")

    gdf = clean_columns(gpd.read_file(path))
    if gdf.crs is None:
        raise ValueError(f"{path} does not have a coordinate reference system")

    if geography_type == "county":
        gdf["CO_FIPS"] = pd.to_numeric(gdf["FIPS"], errors="coerce").astype("Int64")
        gdf["CO_NAME"] = gdf["NAME"].astype(str).str.upper()
    else:
        gdf = normalize_int_columns(gdf, ["DISTLRG", "DISTMED", "DISTSML"])

    gdf = add_feature_ids(gdf, geography_type, config["keys"], config["name"])
    keep_cols = list(dict.fromkeys(config["keys"] + [config["name"], "geography_type", "geography_id", "geography_name", "geometry"]))
    return add_area_sqmi(gdf[keep_cols]).to_crs(epsg=WEB_MAP_EPSG)


def load_year_table(se_dir, year, geo_xwalk, round_metrics=True):
    se = clean_columns(pd.read_csv(se_dir / f"SE_{year}.csv")).rename(columns={"TAZID": "taz_id"})
    se["year"] = year
    se["taz_id"] = pd.to_numeric(se["taz_id"], errors="coerce").astype("Int64")
    if "CO_TAZID" in se.columns:
        se["CO_TAZID"] = pd.to_numeric(se["CO_TAZID"], errors="coerce").astype("Int64")

    drop_geo = [col for col in GEO_FIELDS if col in se.columns and col != "CO_TAZID"]
    se = se.drop(columns=drop_geo).merge(geo_xwalk, on="taz_id", how="left", suffixes=("", "_shp"))
    if "CO_TAZID_shp" in se.columns:
        se["CO_TAZID"] = se["CO_TAZID"].combine_first(se["CO_TAZID_shp"])
        se = se.drop(columns=["CO_TAZID_shp"])

    for metric in METRICS:
        if metric not in se.columns:
            se[metric] = 0
        se[metric] = pd.to_numeric(se[metric], errors="coerce").fillna(0)
        if round_metrics:
            se[metric] = se[metric].round(3)
    se["HHSIZE"] = se["HHPOP"].div(se["TOTHH"].replace(0, pd.NA)).fillna(0)
    if round_metrics:
        se["HHSIZE"] = se["HHSIZE"].round(3)
    return se


def geography_rows(se, geography_type):
    config = GEOGRAPHIES[geography_type]
    keys = config["keys"]
    name_col = config["name"]

    if geography_type == "taz":
        label_cols = [col for col in GEO_FIELDS if col not in ["TAZID", "CO_TAZID"]]
        rows = se[["year", "taz_id", "CO_TAZID"] + label_cols + METRICS].copy()
        rows["geography_type"] = "taz"
        rows["geography_id"] = rows["taz_id"].astype(str)
        rows["geography_name"] = "TAZ " + rows["geography_id"]
        return rows

    valid = se.dropna(subset=keys).copy()
    valid = valid[valid[keys[-1]].astype(str).str.strip().ne("")]
    grouped = valid.groupby(keys, dropna=False)[METRICS].sum().reset_index()
    grouped["HHSIZE"] = grouped["HHPOP"].div(grouped["TOTHH"].replace(0, pd.NA)).fillna(0).round(3)

    if name_col not in grouped.columns:
        grouped = grouped.merge(valid[keys + [name_col]].drop_duplicates(keys), on=keys, how="left")
    grouped["year"] = int(se["year"].iloc[0])
    grouped["taz_id"] = pd.NA
    grouped["CO_TAZID"] = pd.NA
    grouped["geography_type"] = geography_type
    grouped["geography_id"] = grouped[keys].astype(str).agg("_".join, axis=1)
    grouped["geography_name"] = grouped[name_col].fillna(grouped["geography_id"])

    label_cols = [field for field in GEO_FIELDS if field not in ["TAZID", "CO_TAZID"]]
    for col in label_cols:
        if col not in grouped.columns:
            grouped[col] = pd.NA
    return grouped[["year", "taz_id", "CO_TAZID"] + label_cols + ["geography_type", "geography_id", "geography_name"] + METRICS]


def build_forecast_table(se_dir, years, geo_xwalk):
    parts = []
    for year in years:
        se = load_year_table(se_dir, year, geo_xwalk)
        for geography_type in GEOGRAPHIES:
            parts.append(geography_rows(se, geography_type))

    forecast = pd.concat(parts, ignore_index=True)
    first_cols = ["year", "geography_type", "geography_id", "geography_name", "taz_id", "CO_TAZID"]
    label_cols = [col for col in GEO_FIELDS if col not in ["TAZID", "CO_TAZID"]]
    return forecast[first_cols + label_cols + METRICS].sort_values(["geography_type", "geography_id", "year"])


def build_change_table(forecast, years):
    id_cols = ["geography_type", "geography_id", "geography_name"]
    start = forecast.loc[forecast["year"].eq(years[0]), id_cols + METRICS]
    end = forecast.loc[forecast["year"].eq(years[-1]), id_cols + METRICS]
    change = start.merge(end, on=id_cols, how="outer", suffixes=("_from", "_to")).fillna(0)
    change["from_year"] = years[0]
    change["to_year"] = years[-1]

    for metric in METRICS:
        change[f"{metric}_change"] = change[f"{metric}_to"] - change[f"{metric}_from"]
        change[f"{metric}_pct_change"] = (
            change[f"{metric}_change"].div(change[f"{metric}_from"].replace(0, pd.NA)).mul(100).fillna(0)
        )
    return change


def preview_label_columns():
    return [col for col in GEO_FIELDS if col not in ["TAZID"]]


def add_preview_metrics(df):
    df = df.copy()
    for alias, source in PREVIEW_METRIC_ALIASES.items():
        df[alias] = df[source]
    df["hh_job_total"] = df["households"] + df["jobs_total"]
    df["hh_size"] = df["population"].div(df["households"].replace(0, pd.NA)).fillna(0)
    return df


def preview_geography_rows(se, geography_type):
    label_cols = preview_label_columns()
    se = add_preview_metrics(se)

    if geography_type == "taz":
        rows = se[["year", "taz_id"] + label_cols + PREVIEW_METRICS].copy()
        rows["geography_type"] = "taz"
        rows["geography_id"] = rows["taz_id"].astype(str)
        rows["geography_name"] = "TAZ " + rows["geography_id"]
        return rows[
            ["year", "geography_type", "geography_id", "geography_name", "taz_id"]
            + label_cols
            + PREVIEW_METRICS
        ]

    config = PREVIEW_GEOGRAPHIES[geography_type]
    keys = config["keys"]
    name_col = config["name"]
    valid = se.dropna(subset=keys).copy()
    valid = valid[valid[keys[-1]].astype(str).str.strip().ne("")]

    grouped = valid.groupby(keys, dropna=False)[list(PREVIEW_METRIC_ALIASES)].sum().reset_index()
    grouped["hh_job_total"] = grouped["households"] + grouped["jobs_total"]
    grouped["hh_size"] = grouped["population"].div(grouped["households"].replace(0, pd.NA)).fillna(0)
    grouped["year"] = int(se["year"].iloc[0])
    grouped["taz_id"] = pd.NA
    grouped["geography_type"] = geography_type
    grouped["geography_id"] = grouped[keys].astype(str).agg("_".join, axis=1)
    if name_col not in grouped.columns:
        grouped = grouped.merge(valid[keys + [name_col]].drop_duplicates(keys), on=keys, how="left")
    grouped["geography_name"] = grouped[name_col].fillna(grouped["geography_id"])

    for col in label_cols:
        if col not in grouped.columns:
            grouped[col] = pd.NA
    return grouped[
        ["year", "geography_type", "geography_id", "geography_name", "taz_id"]
        + label_cols
        + PREVIEW_METRICS
    ]


def build_preview_forecast_table(se_dir, years, geo_xwalk):
    parts = []
    for year in years:
        se = load_year_table(se_dir, year, geo_xwalk, round_metrics=False)
        for geography_type in PREVIEW_GEOGRAPHIES:
            parts.append(preview_geography_rows(se, geography_type))

    forecast = pd.concat(parts, ignore_index=True)
    return forecast.sort_values(["geography_type", "geography_id", "year"])


def build_preview_change_table(forecast, years):
    id_cols = ["geography_type", "geography_id", "geography_name"]
    start = forecast.loc[forecast["year"].eq(years[0]), id_cols + PREVIEW_METRICS]
    end = forecast.loc[forecast["year"].eq(years[-1]), id_cols + PREVIEW_METRICS]
    change = start.merge(end, on=id_cols, how="outer", suffixes=("_from", "_to")).fillna(0)
    change["from_year"] = years[0]
    change["to_year"] = years[-1]
    return change


def build_preview_change_long(change):
    rows = []
    id_cols = ["geography_type", "geography_id", "geography_name", "from_year", "to_year"]
    for metric in PREVIEW_METRICS:
        part = change[id_cols + [f"{metric}_from", f"{metric}_to"]].copy()
        part = part.rename(columns={f"{metric}_from": "from_value", f"{metric}_to": "to_value"})
        part["metric"] = metric
        part["absolute_change"] = part["to_value"] - part["from_value"]
        part["percent_change"] = (
            part["absolute_change"].div(part["from_value"].replace(0, pd.NA)).mul(100).fillna(0)
        )
        rows.append(part)
    return pd.concat(rows, ignore_index=True)[
        id_cols + ["from_value", "to_value", "metric", "absolute_change", "percent_change"]
    ]


def build_preview_trends_long(forecast):
    id_cols = ["year", "geography_type", "geography_id", "geography_name"]
    return forecast.melt(id_vars=id_cols, value_vars=PREVIEW_METRICS, var_name="metric", value_name="value")


def ensure_preview_page_controls():
    if not INDEX_HTML.exists():
        return

    html = INDEX_HTML.read_text(encoding="utf-8")
    original = html
    if 'id="basemapToggle"' not in html:
        html = html.replace(
            '      <div class="checks">\n',
            '      <div class="checks">\n'
            '        <label><input type="checkbox" id="basemapToggle" checked> Show Google Map Background</label>\n',
            1,
        )
    if "const googleBaseLayer = L.tileLayer" not in html:
        html = html.replace(
            '    L.tileLayer("https://mt1.google.com/vt/lyrs=m&x={x}&y={y}&z={z}", {\n',
            '    const googleBaseLayer = L.tileLayer("https://mt1.google.com/vt/lyrs=m&x={x}&y={y}&z={z}", {\n',
            1,
        )
    if 'document.getElementById("basemapToggle").addEventListener("change"' not in html:
        html = html.replace(
            '      document.getElementById("countyToggle").addEventListener("change", event => refreshOverlay("county", event.target.checked));\n',
            '      document.getElementById("basemapToggle").addEventListener("change", event => {\n'
            '        if (event.target.checked) googleBaseLayer.addTo(map);\n'
            '        else map.removeLayer(googleBaseLayer);\n'
            '      });\n'
            '      document.getElementById("countyToggle").addEventListener("change", event => refreshOverlay("county", event.target.checked));\n',
            1,
        )
    if 'const zeroColor = "#b8b8b8";' not in html:
        html = html.replace(
            '    const mapColors = ["#064b1f","#197337","#67a545","#d9c95b","#e8913d","#c43b32","#7f0000"];\n',
            '    const zeroColor = "#b8b8b8";\n'
            '    const mapColors = [zeroColor,"#064b1f","#197337","#d9c95b","#e8913d","#c43b32","#7f0000"];\n',
            1,
        )
    if 'const mapColors = ["#064b1f"' not in html and 'const mapColors = [zeroColor' not in html:
        html = html.replace(
            '    const chartColors = ["#e4a300","#0a7d22","#7a62d9","#2a63df","#8c1e14","#c24ccf","#777","#111","#b25f00","#2f80ed","#875a37","#426b1f","#99582a","#437f97","#89023e","#d00000","#6a994e","#577590","#bc6c25","#00a99d"];\n',
            '    const chartColors = ["#e4a300","#0a7d22","#7a62d9","#2a63df","#8c1e14","#c24ccf","#777","#111","#b25f00","#2f80ed","#875a37","#426b1f","#99582a","#437f97","#89023e","#d00000","#6a994e","#577590","#bc6c25","#00a99d"];\n'
            '    const zeroColor = "#b8b8b8";\n'
            '    const mapColors = [zeroColor,"#064b1f","#197337","#d9c95b","#e8913d","#c43b32","#7f0000"];\n',
            1,
        )
    if "if (value === 0) return zeroColor;" not in html:
        html = html.replace(
            "    function colorScale(value, breaks) {\n"
            "      const colors = mapColors;\n",
            "    function colorScale(value, breaks) {\n"
            "      if (value === 0) return zeroColor;\n"
            "      const colors = mapColors;\n",
            1,
        )
    if "visibleMetrics: new Set(metrics)" not in html:
        html = html.replace(
            "      chart: null\n",
            "      chart: null,\n"
            "      visibleMetrics: new Set(metrics)\n",
            1,
        )
    html = html.replace(
        '      const forecastColors = ["#fff8e5","#eadcab","#d6bf6b","#bd982f","#9f751a","#7e5310","#5c3407"];\n'
        '      const changeColors = ["#cfe3ea","#9eb7c1","#ddd1d0","#c98688","#ad4f59","#8e2536","#671227"];\n'
        '      const colors = state.mode === "forecast" ? forecastColors : changeColors;\n',
        "      const colors = mapColors;\n",
    )
    html = html.replace(
        '      const colors = state.mode === "forecast"\n'
        '        ? ["#fff8e5","#eadcab","#d6bf6b","#bd982f","#9f751a","#7e5310","#5c3407"]\n'
        '        : ["#cfe3ea","#9eb7c1","#ddd1d0","#c98688","#ad4f59","#8e2536","#671227"];\n',
        "      const colors = mapColors;\n",
    )
    if ".series input { margin: 0; }" not in html:
        html = html.replace(
            "    .series label { font-weight: 400; }\n",
            "    .series label { font-weight: 400; display: flex; align-items: center; gap: 4px; min-width: 0; }\n"
            "    .series input { margin: 0; }\n",
            1,
        )
    if ".status-fill" not in html:
        html = html.replace(
            "    .status { position: absolute; z-index: 500; left: 384px; bottom: 12px; background: rgba(0,0,0,.8); color: white; padding: 4px 8px; font-size: 11px; }\n",
            "    .status { position: absolute; z-index: 500; left: 382px; bottom: 12px; width: 230px; background: rgba(0,0,0,.82); color: white; padding: 7px 8px; font-size: 11px; box-sizing: border-box; transition: opacity .2s ease; }\n"
            "    .status.hidden { opacity: 0; pointer-events: none; }\n"
            "    .status-text { display: flex; justify-content: space-between; gap: 8px; margin-bottom: 5px; }\n"
            "    .status-bar { height: 5px; background: rgba(255,255,255,.22); overflow: hidden; }\n"
            "    .status-fill { display: block; height: 100%; width: 0; background: #83ea89; transition: width .15s ease; }\n",
            1,
        )
        html = html.replace(
            '  <div class="status" id="status">Loading...</div>\n',
            '  <div class="status" id="status">\n'
            '    <div class="status-text"><span id="statusText">Loading</span><span id="statusPct">0%</span></div>\n'
            '    <div class="status-bar"><span class="status-fill" id="statusFill"></span></div>\n'
            '  </div>\n',
            1,
        )
    if "function setStatus(message, percent = null)" not in html:
        html = html.replace(
            "    function numeric(row, key) {\n",
            '    function setStatus(message, percent = null) {\n'
            '      const status = document.getElementById("status");\n'
            "      if (!status) return;\n"
            '      status.classList.remove("hidden");\n'
            "      const pct = percent === null ? null : Math.max(0, Math.min(100, Math.round(percent)));\n"
            '      document.getElementById("statusText").textContent = message;\n'
            '      document.getElementById("statusPct").textContent = pct === null ? "" : `${pct}%`;\n'
            '      document.getElementById("statusFill").style.width = pct === null ? "0%" : `${pct}%`;\n'
            "    }\n"
            "    function hideStatus(delay = 900) {\n"
            '      window.setTimeout(() => document.getElementById("status")?.classList.add("hidden"), delay);\n'
            "    }\n"
            "    async function fetchTextWithProgress(url, label, startPercent, endPercent) {\n"
            "      setStatus(label, startPercent);\n"
            "      const response = await fetch(url);\n"
            '      if (!response.ok) throw new Error(`Failed to load ${url}: ${response.status}`);\n'
            '      const total = Number(response.headers.get("content-length"));\n'
            "      if (!response.body || !Number.isFinite(total) || total <= 0) {\n"
            "        const text = await response.text();\n"
            "        setStatus(label, endPercent);\n"
            "        return text;\n"
            "      }\n"
            "      const reader = response.body.getReader();\n"
            "      const decoder = new TextDecoder();\n"
            "      const chunks = [];\n"
            "      let loaded = 0;\n"
            "      while (true) {\n"
            "        const { done, value } = await reader.read();\n"
            "        if (done) break;\n"
            "        loaded += value.byteLength;\n"
            "        chunks.push(decoder.decode(value, { stream: true }));\n"
            "        setStatus(label, startPercent + (loaded / total) * (endPercent - startPercent));\n"
            "      }\n"
            "      chunks.push(decoder.decode());\n"
            "      setStatus(label, endPercent);\n"
            '      return chunks.join("");\n'
            "    }\n"
            "    function numeric(row, key) {\n",
            1,
        )
    html = html.replace(
        "    async function loadFeatureLayer(type) {\n"
        "      if (state.features[type]) return;\n"
        "      state.features[type] = await fetch(layerFiles[type]).then(response => response.json());\n"
        "      indexFeatureAreaCollection(type);\n"
        "    }\n"
        "    async function loadForecastRows(type) {\n"
        "      if (state.loadedDataGeographies.has(type)) return;\n"
        '      const text = await fetch(forecastFiles[type] || "data/forecast.csv").then(response => response.text());\n',
        "    async function loadFeatureLayer(type, startPercent = 0, endPercent = 50) {\n"
        "      if (state.features[type]) return;\n"
        '      const label = `Loading ${geographyLabels[type] || type} shapes`;\n'
        "      state.features[type] = JSON.parse(await fetchTextWithProgress(layerFiles[type], label, startPercent, endPercent));\n"
        "      indexFeatureAreaCollection(type);\n"
        "    }\n"
        "    async function loadForecastRows(type, startPercent = 50, endPercent = 95) {\n"
        "      if (state.loadedDataGeographies.has(type)) return;\n"
        '      const label = `Loading ${geographyLabels[type] || type} data`;\n'
        '      const text = await fetchTextWithProgress(forecastFiles[type] || "data/forecast.csv", label, startPercent, endPercent);\n',
    )
    html = html.replace(
        "    async function ensureGeography(type) {\n"
        "      await Promise.all([loadFeatureLayer(type), loadForecastRows(type)]);\n"
        "    }\n",
        "    async function ensureGeography(type) {\n"
        "      await loadFeatureLayer(type, 5, 60);\n"
        "      await loadForecastRows(type, 60, 96);\n"
        "    }\n",
    )
    html = html.replace(
        '      document.getElementById("geoSelect").addEventListener("change", async event => {\n'
        '        const status = document.getElementById("status");\n'
        "        state.geography = event.target.value;\n"
        "        state.selectedKey = null;\n"
        "        status.textContent = `Loading ${geographyLabels[state.geography]}...`;\n"
        "        await ensureGeography(state.geography);\n"
        "        drawActiveLayer();\n"
        "        map.fitBounds(state.activeLayer.getBounds(), { padding: [10, 10] });\n"
        '        status.textContent = "Loaded";\n'
        "      });\n",
        '      document.getElementById("geoSelect").addEventListener("change", async event => {\n'
        "        state.geography = event.target.value;\n"
        "        state.selectedKey = null;\n"
        '        setStatus(`Loading ${geographyLabels[state.geography]}`, 0);\n'
        "        await ensureGeography(state.geography);\n"
        "        drawActiveLayer();\n"
        "        map.fitBounds(state.activeLayer.getBounds(), { padding: [10, 10] });\n"
        '        setStatus("Loaded", 100);\n'
        "        hideStatus();\n"
        "      });\n",
    )
    html = html.replace(
        "    async function init() {\n"
        '      const status = document.getElementById("status");\n'
        '      const metadata = await fetch("data/metadata.json").then(response => response.json());\n',
        "    async function init() {\n"
        '      setStatus("Loading metadata", 0);\n'
        '      const metadata = await fetch("data/metadata.json").then(response => response.json());\n',
    )
    html = html.replace(
        '      status.textContent = "Loaded";\n'
        "      setTimeout(() => status.remove(), 1200);\n",
        '      setStatus("Loaded", 100);\n'
        "      hideStatus();\n",
    )
    html = html.replace(
        '      document.getElementById("status").textContent = `Error: ${error.message}`;\n',
        "      setStatus(`Error: ${error.message}`);\n",
    )
    if "hidden: !state.visibleMetrics.has(metric)" not in html:
        html = html.replace(
            "        backgroundColor: chartColors[i],\n",
            "        backgroundColor: chartColors[i],\n"
            "        hidden: !state.visibleMetrics.has(metric),\n",
            1,
        )
    if "state.visibleMetrics.add(metric)" not in html:
        html = html.replace(
            '        label.innerHTML = `<span class="swatch" style="background:${chartColors[i]}"></span>${metricLabel(metric)}`;\n'
            "        series.appendChild(label);\n",
            '        const input = document.createElement("input");\n'
            '        input.type = "checkbox";\n'
            '        input.checked = state.visibleMetrics.has(metric);\n'
            '        input.addEventListener("change", event => {\n'
            '          if (event.target.checked) state.visibleMetrics.add(metric);\n'
            '          else state.visibleMetrics.delete(metric);\n'
            '          refreshChart();\n'
            '        });\n'
            '        const swatch = document.createElement("span");\n'
            '        swatch.className = "swatch";\n'
            '        swatch.style.background = chartColors[i];\n'
            '        const text = document.createElement("span");\n'
            '        text.textContent = metricLabel(metric);\n'
            '        label.append(input, swatch, text);\n'
            "        series.appendChild(label);\n",
            1,
        )
    if html != original:
        INDEX_HTML.write_text(html, encoding="utf-8")


def main():
    ensure_preview_page_controls()
    DOCS_DATA_DIR.mkdir(parents=True, exist_ok=True)
    se_dir, start_year, end_year = review_config()
    years = available_years(se_dir, start_year, end_year)
    taz_gdf, taz_map_gdf, geo_xwalk = build_taz_geography()
    layers = {"taz": taz_gdf}
    for geography_type in ["city", "county", "large_district", "medium_district", "small_district"]:
        layers[geography_type] = read_boundary_layer(geography_type)

    forecast = build_forecast_table(se_dir, years, geo_xwalk)
    change = build_change_table(forecast, years)
    preview_forecast = build_preview_forecast_table(se_dir, years, geo_xwalk)
    preview_change = build_preview_change_table(preview_forecast, years)
    preview_change_long = build_preview_change_long(preview_change)
    preview_trends_long = build_preview_trends_long(preview_forecast)

    files = {}
    for geography_type, gdf in layers.items():
        file_name = GEOGRAPHIES[geography_type]["file"]
        gdf.to_file(DOCS_DATA_DIR / file_name, driver="GeoJSON")
        files[f"{geography_type}_features"] = f"data/{file_name}"
    taz_map_gdf.to_file(DOCS_DATA_DIR / TAZ_GEOJSON_FILE, driver="GeoJSON")

    forecast.to_csv(DOCS_DATA_DIR / "forecast.csv", index=False)
    change.to_csv(DOCS_DATA_DIR / "change.csv", index=False)
    files["forecast"] = "data/forecast.csv"
    files["change"] = "data/change.csv"
    for geography_type, rows in forecast.groupby("geography_type", sort=False):
        path = DOCS_DATA_DIR / f"forecast_{geography_type}.csv"
        rows.to_csv(path, index=False)

    preview_forecast.to_csv(DOCS_DATA_DIR / PREVIEW_FILES["forecast_map_data"], index=False)
    preview_change.to_csv(DOCS_DATA_DIR / PREVIEW_FILES["change_map_data"], index=False)
    preview_change_long.to_csv(DOCS_DATA_DIR / PREVIEW_FILES["change_map_data_long"], index=False)
    preview_trends_long.to_csv(DOCS_DATA_DIR / PREVIEW_FILES["clicked_geography_trends_long"], index=False)
    (DOCS_DATA_DIR / "map_data_dictionary.json").write_text(
        json.dumps(
            {
                "source_directory": str(se_dir),
                "taz_shapefile": str(TAZ_SHP),
                "years": years,
                "geographies": list(PREVIEW_GEOGRAPHIES),
                "geography_fields_from_taz_shp": GEO_FIELDS,
                "metrics": PREVIEW_METRIC_ALIASES,
                "files": {
                    "taz_map_features": str(Path("docs") / "data" / TAZ_GEOJSON_FILE),
                    "forecast_map_data": str(Path("docs") / "data" / PREVIEW_FILES["forecast_map_data"]),
                    "change_map_data": str(Path("docs") / "data" / PREVIEW_FILES["change_map_data"]),
                    "change_map_data_long": str(Path("docs") / "data" / PREVIEW_FILES["change_map_data_long"]),
                    "clicked_geography_trends_long": str(
                        Path("docs") / "data" / PREVIEW_FILES["clicked_geography_trends_long"]
                    ),
                },
            },
            indent=2,
        ),
        encoding="utf-8",
    )

    (DOCS_DATA_DIR / "metadata.json").write_text(
        json.dumps(
            {
                "source_directory": str(se_dir),
                "taz_shapefile": str(TAZ_SHP),
                "districts_directory": str(DISTRICTS_DIR),
                "county_shapefile": str(COUNTY_SHP),
                "years": years,
                "trend_years": [year for year in years if year == years[0] or year == years[-1] or year % 5 == 0],
                "geographies": list(GEOGRAPHIES),
                "metrics": METRICS,
                "files": files,
            },
            indent=2,
        ),
        encoding="utf-8",
    )

    for geography_type, gdf in layers.items():
        print(f"Wrote {len(gdf):,} {geography_type} features")
    print(f"Wrote {len(taz_map_gdf):,} TAZ map features")
    print(f"Wrote {len(forecast):,} forecast rows")
    print(f"Wrote {len(change):,} change rows")
    print(f"Wrote {len(preview_forecast):,} preview forecast rows")
    print(f"Wrote {len(preview_change):,} preview change rows")
    print(f"Wrote {len(preview_change_long):,} preview change-long rows")
    print(f"Wrote {len(preview_trends_long):,} preview trend rows")


if __name__ == "__main__":
    main()


Wrote 3,546 taz features
Wrote 109 city features
Wrote 29 county features
Wrote 26 large_district features
Wrote 73 medium_district features
Wrote 129 small_district features
Wrote 3,546 TAZ map features
Wrote 167,184 forecast rows
Wrote 3,888 change rows
Wrote 169,549 preview forecast rows
Wrote 3,943 preview change rows
Wrote 82,803 preview change-long rows
Wrote 3,560,529 preview trend rows


# review.ipynb
 
 Combined from `Archived/python/review.ipynb`.

In [2]:
from pathlib import Path
import json
import re
import warnings

import geopandas as gpd
import pandas as pd
import yaml

warnings.filterwarnings("ignore", category=FutureWarning)

# Generate map-ready data for the later Forecast and Change map views.
# Paths are resolved from the ResultReview repo root so this cell works when
# launched from either the repo root or the python notebook folder.
cwd = Path.cwd().resolve()
repo_dir = cwd.parent if cwd.name.lower() == "python" else cwd
config_path = repo_dir / "_config.yml"
config = yaml.safe_load(config_path.read_text(encoding="utf-8"))["review_pages"]


def config_path_value(value, base=repo_dir):
    path = Path(value)
    return path if path.is_absolute() else base / path


input_files_dir = config_path_value(config["input_files_dir"]).resolve()
se_dir = config_path_value(config["se_data_dir"], input_files_dir).resolve()
taz_shp = config_path_value(config["taz_shapefile"], input_files_dir).resolve()
output_dir = config_path_value(config["docs_data_dir"]).resolve()
output_dir.mkdir(parents=True, exist_ok=True)


def clean_columns(df):
    df = df.copy()
    df.columns = [str(col).lstrip(";").strip() for col in df.columns]
    return df


def repo_relative(path):
    path = Path(path)
    try:
        return str(path.resolve().relative_to(repo_dir))
    except ValueError:
        return str(path)


if not se_dir.exists():
    raise FileNotFoundError(f"SE data directory not found: {se_dir}")
if not taz_shp.exists():
    raise FileNotFoundError(f"TAZ shapefile not found: {taz_shp}")

taz_gdf = clean_columns(gpd.read_file(taz_shp))
taz_fields = config["geography_fields"]
for col in taz_fields:
    if col not in taz_gdf.columns:
        taz_gdf[col] = pd.NA

geo_xwalk = taz_gdf[taz_fields].copy()
for col in ["TAZID", "CO_TAZID", "CO_FIPS", "CITY_FIPS", "DISTLRG", "DISTMED", "DISTSML"]:
    geo_xwalk[col] = pd.to_numeric(geo_xwalk[col], errors="coerce").astype("Int64")
geo_xwalk = geo_xwalk.drop_duplicates("TAZID")

# Keep one geometry file ready for the future TAZ map layer.
taz_geojson_path = output_dir / config["taz_geojson_file"]
taz_gdf[taz_fields + ["geometry"]].to_file(taz_geojson_path, driver="GeoJSON")

metrics = config["preview_metric_aliases"]

label_cols = [field for field in taz_fields if field not in ["TAZID", "CO_TAZID"]]
metric_cols = list(metrics) + ["hh_job_total", "hh_size"]

years = []
for path in sorted(se_dir.glob("SE_*.csv")):
    match = re.search(r"SE_(\d{4})\.csv$", path.name)
    if match:
        years.append(int(match.group(1)))
if not years:
    raise FileNotFoundError(f"No SE_YYYY.csv files found in {se_dir}")

forecast_parts = []
for year in years:
    se = clean_columns(pd.read_csv(se_dir / f"SE_{year}.csv")).rename(columns={"TAZID": "taz_id"})
    se["year"] = year
    se["taz_id"] = pd.to_numeric(se["taz_id"], errors="coerce").astype("Int64")
    if "CO_TAZID" in se.columns:
        se["CO_TAZID"] = pd.to_numeric(se["CO_TAZID"], errors="coerce").astype("Int64")

    drop_geo = [col for col in taz_fields if col in se.columns and col != "CO_TAZID"]
    se = se.drop(columns=drop_geo).merge(
        geo_xwalk,
        left_on="taz_id",
        right_on="TAZID",
        how="left",
        suffixes=("", "_shp"),
    )
    if "CO_TAZID_shp" in se.columns:
        se["CO_TAZID"] = se["CO_TAZID"].combine_first(se["CO_TAZID_shp"])
        se = se.drop(columns=["CO_TAZID_shp"])

    for out_col, src_col in metrics.items():
        if src_col not in se.columns:
            se[src_col] = 0
        se[out_col] = pd.to_numeric(se[src_col], errors="coerce").fillna(0)
    se["hh_job_total"] = se["households"] + se["jobs_total"]
    se["hh_size"] = se["population"].div(se["households"].replace(0, pd.NA)).fillna(0)

    common = ["year", "taz_id", "CO_TAZID"] + label_cols
    taz = se[common + metric_cols].copy()
    taz["geography_type"] = "taz"
    taz["geography_id"] = taz["taz_id"].astype(str)
    taz["geography_name"] = "TAZ " + taz["geography_id"]
    forecast_parts.append(taz)

    for geography_type, geography_config in config["preview_geographies"].items():
        if geography_type == "taz":
            continue
        keys = geography_config["keys"]
        name_col = geography_config["name"]
        valid = se.dropna(subset=keys).copy()
        valid = valid[valid[keys[-1]].astype(str).str.strip().ne("")]
        if valid.empty:
            continue
        grouped = valid.groupby(keys, dropna=False)[metric_cols].sum().reset_index()
        if name_col not in grouped.columns:
            grouped = grouped.merge(valid[keys + [name_col]].drop_duplicates(keys), on=keys, how="left")
        grouped["hh_job_total"] = grouped["households"] + grouped["jobs_total"]
        grouped["hh_size"] = grouped["population"].div(grouped["households"].replace(0, pd.NA)).fillna(0)
        grouped["year"] = year
        grouped["geography_type"] = geography_type
        grouped["geography_id"] = grouped[keys].astype(str).agg("_".join, axis=1)
        grouped["geography_name"] = grouped[name_col].fillna(grouped["geography_id"])
        for col in common:
            if col not in grouped.columns:
                grouped[col] = pd.NA
        forecast_parts.append(grouped[common + ["geography_type", "geography_id", "geography_name"] + metric_cols])

forecast = pd.concat(forecast_parts, ignore_index=True)
id_cols = ["geography_type", "geography_id", "geography_name"]
forecast = forecast[["year"] + id_cols + ["taz_id", "CO_TAZID"] + label_cols + metric_cols].sort_values(id_cols + ["year"])

start = forecast.loc[forecast["year"].eq(years[0]), id_cols + metric_cols]
end = forecast.loc[forecast["year"].eq(years[-1]), id_cols + metric_cols]
change = start.merge(end, on=id_cols, how="outer", suffixes=("_from", "_to")).fillna(0)
change["from_year"] = years[0]
change["to_year"] = years[-1]

change_long_parts = []
for metric in metric_cols:
    part = change[id_cols + ["from_year", "to_year", f"{metric}_from", f"{metric}_to"]].copy()
    part = part.rename(columns={f"{metric}_from": "from_value", f"{metric}_to": "to_value"})
    part["metric"] = metric
    part["absolute_change"] = part["to_value"] - part["from_value"]
    part["percent_change"] = part["absolute_change"].div(part["from_value"].replace(0, pd.NA)).mul(100).fillna(0)
    change_long_parts.append(part)
change_long = pd.concat(change_long_parts, ignore_index=True)

trends_long = forecast.melt(id_vars=["year"] + id_cols, value_vars=metric_cols, var_name="metric", value_name="value")

preview_files = config["preview_files"]
forecast_path = output_dir / preview_files["forecast_map_data"]
change_path = output_dir / preview_files["change_map_data"]
change_long_path = output_dir / preview_files["change_map_data_long"]
trends_path = output_dir / preview_files["clicked_geography_trends_long"]
dictionary_path = output_dir / "map_data_dictionary.json"

forecast.to_csv(forecast_path, index=False)
change.to_csv(change_path, index=False)
change_long.to_csv(change_long_path, index=False)
trends_long.to_csv(trends_path, index=False)
dictionary_path.write_text(json.dumps({
    "source_directory": repo_relative(se_dir),
    "taz_shapefile": repo_relative(taz_shp),
    "years": years,
    "geographies": list(config["preview_geographies"]),
    "geography_fields_from_taz_shp": taz_fields,
    "metrics": metrics,
    "files": {
        "taz_map_features": repo_relative(taz_geojson_path),
        "forecast_map_data": repo_relative(forecast_path),
        "change_map_data": repo_relative(change_path),
        "change_map_data_long": repo_relative(change_long_path),
        "clicked_geography_trends_long": repo_relative(trends_path),
    },
}, indent=2), encoding="utf-8")

print(f"Read SE data from {repo_relative(se_dir)}")
print(f"Read geography fields from {repo_relative(taz_shp)}")
print(f"Wrote TAZ geometry/features to {repo_relative(taz_geojson_path)}")
print(f"Wrote {len(forecast):,} forecast rows to {repo_relative(forecast_path)}")
print(f"Wrote {len(change):,} change rows to {repo_relative(change_path)}")
print(f"Wrote {len(change_long):,} long change rows to {repo_relative(change_long_path)}")
print(f"Wrote {len(trends_long):,} clicked-geography trend rows to {repo_relative(trends_path)}")

Read SE data from E:\xtemp\GitStuff\REMM-v3.0\TDM\_TDMv9.0.0_REMM\1_Inputs\2_SEData\REMM
Read geography fields from E:\xtemp\GitStuff\REMM-v3.0\TDM\_TDMv9.0.0_REMM\1_Inputs\1_TAZ\TAZ.shp
Wrote TAZ geometry/features to docs\data\taz_map_features.geojson
Wrote 169,549 forecast rows to docs\data\forecast_map_data.csv
Wrote 3,943 change rows to docs\data\change_map_data.csv
Wrote 82,803 long change rows to docs\data\change_map_data_long.csv
Wrote 3,560,529 clicked-geography trend rows to docs\data\clicked_geography_trends_long.csv
